In [ ]:
def tau_and_capacitance(file, voltage_channel, pre_start, post_stop, Rin_rel, search_range=(-50, -10), sampling_rate=10000):
    """
    Находит подходящий свип с отрицательным изменением напряжения,
    вычисляет мембранную постоянную времени (tau) и емкость клетки.

    Parameters:
        file : ABF file object
        voltage_channel : int
        pre_start : int, индекс начала до стимуляции
        post_stop : int, индекс конца после стимуляции
        Rin_rel : float, входное сопротивление в МОм
        search_range : tuple, допустимый диапазон изменения напряжения (мВ)
        sampling_rate : int, Гц

    Returns:
        tau_ms : float, мембранная постоянная времени в мс
        capacitance_pF : float, емкость клетки в пФ
        chosen_sweep : int, номер выбранного свипа
    """
    best_sweep = None
    min_delta = np.inf
    
    # Поиск свипа с нужным отрицательным ΔV
    for sweep_num in range(len(file.sweepList)):
        file.setSweep(sweep_num, channel=voltage_channel)
        voltage_pre = np.mean(file.sweepY[pre_start:pre_start + int(0.125*sampling_rate)])
        voltage_min = np.min(file.sweepY[pre_start:post_stop])
        delta_v = voltage_min - voltage_pre
        
        if search_range[0] <= delta_v <= search_range[1]:
            if abs(delta_v) < min_delta:  # ближе к нулю
                min_delta = abs(delta_v)
                best_sweep = sweep_num
                trace = file.sweepY[pre_start:post_stop]
                v_pre = voltage_pre
                v_min = voltage_min

    if best_sweep is None:
        print("Свип с нужным отрицательным ΔV не найден")
        return np.nan, np.nan, None

    # Вычисление tau: время достижения 63% ΔV
    v_63 = v_pre + 0.63 * (v_min - v_pre)
    for i, v in enumerate(trace):
        if v <= v_63:  # мембранный потенциал достигает 63% изменения
            tau_samples = i
            break
    tau_ms = (tau_samples / sampling_rate) * 1000  # в мс

    # Емкость: tau / Rin
    if Rin_abs is not None and not np.isnan(Rin_abs):
        capacitance_pF = (tau_ms / Rin_abs) * 1000  # МОм * мс -> пФ
    else:
        capacitance_pF = np.nan

    return tau_ms, capacitance_pF, best_sweep